# Preprocessing 04: Data Partitioning and Leakage Prevention

In [1]:
# import libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, brier_score_loss
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

In [2]:
# Load the analysis table saved by initial_eda_02

shots_analysis_df = pd.read_csv('output/initial_eda_receipt_features.csv', parse_dates=['match_date'])

# Sort by shot id so results match on every computer

shots_analysis_df = shots_analysis_df.sort_values('id').reset_index(drop=True)

print(shots_analysis_df.shape)
print(f"Goals: {shots_analysis_df['is_goal'].sum():,} ({shots_analysis_df['is_goal'].mean():.1%} of shots)")

(37443, 40)
Goals: 3,553 (9.5% of shots)


## Preprocessing for logistic regression

Save the newest 20% of matches for testing. Keep all shots from the same match together. If the cutoff falls on a shared match date, put that whole date in the test set.

Use distance and angle for shot location. Keep the original x and y coordinates for plots. Names, IDs, post-shot fields, and StatsBomb xG stay in the source table. Leave shooter position out for now.

Group shot techniques with fewer than 150 training examples. New techniques go in that same group. Fill missing receipt measurements with 0 and add a flag for missing values.

In [3]:
categorical_features = [
    'shot_type_name', 'shot_body_part_name', 'shot_technique_name', 'play_pattern_name'
]
flag_features = [
    'under_pressure', 'shot_first_time', 'shot_one_on_one',
    'shot_open_goal', 'shot_aerial_won', 'goalkeeper_in_lane'
]
receipt_features = ['receipt_to_shot_seconds', 'receipt_to_shot_displacement']
numeric_features = [
    'shot_distance', 'shot_angle', 'defenders_in_lane',
    'teammates_in_lane', 'percent_of_goal_open', 'goalkeeper_distance_from_goal',
    'goalkeeper_distance_to_shooter', *receipt_features
]
rare_label = 'Rare or unseen'


def fit_shot_preprocessing(training_shots, min_technique_count=150):
    """Learn category groups from training shots and set receipt fill values to zero."""
    technique_counts = training_shots['shot_technique_name'].value_counts()
    common_techniques = set(technique_counts[technique_counts >= min_technique_count].index)
    levels = {
        column: sorted(training_shots[column].dropna().unique())
        for column in categorical_features
    }
    levels['shot_technique_name'] = sorted(common_techniques) + [rare_label]
        
    # Missing receipt values are filled with zero (the missing flag marks them)

    receipt_fill_values = pd.Series(0.0, index=receipt_features)

    return levels, receipt_fill_values


def transform_shot_predictors(shots, fitted):
    """Apply training-derived preprocessing to a set of shots."""
    levels, receipt_medians = fitted
    categories = shots[categorical_features].copy()
    technique = categories['shot_technique_name']
    categories['shot_technique_name'] = technique.where(
        technique.isin(levels['shot_technique_name']), rare_label
    )
    for column in categorical_features:
        # New values in these fields get zeros in all their columns.
        categories[column] = pd.Categorical(categories[column], categories=levels[column])
    encoded = pd.get_dummies(categories, dtype='int8')

    numeric = shots[numeric_features].copy()
    numeric['receipt_measurements_missing'] = numeric[receipt_features].isna().any(axis=1).astype('int8')
    numeric[receipt_features] = numeric[receipt_features].fillna(receipt_medians)
    flags = shots[flag_features].astype('int8')
    return pd.concat([numeric, flags, encoded], axis=1)


### Set aside the test matches

Sort the matches by date and save the newest 20% for the final test. Keep all shots from a match together, and put every match on the cutoff date in the test set.

Use the earlier matches to learn the category groups and fill values. Apply those same rules to the test shots.

In [4]:
match_dates = (
    shots_analysis_df[['match_id', 'match_date']].drop_duplicates()
    .assign(match_date=lambda frame: pd.to_datetime(frame['match_date']))
    .sort_values(['match_date', 'match_id'])
)
assert match_dates['match_id'].is_unique, 'A match has more than one date.'
cutoff_date = match_dates.iloc[len(match_dates) * 4 // 5]['match_date']
test_match_ids = match_dates.loc[match_dates['match_date'] >= cutoff_date, 'match_id']
is_test = shots_analysis_df['match_id'].isin(test_match_ids)
train_shots = shots_analysis_df.loc[~is_test].copy()
test_shots = shots_analysis_df.loc[is_test].copy()

shot_preprocessing = fit_shot_preprocessing(train_shots)
X_train = transform_shot_predictors(train_shots, shot_preprocessing)
X_test = transform_shot_predictors(test_shots, shot_preprocessing)
y_train = train_shots['is_goal'].astype('int8')
y_test = test_shots['is_goal'].astype('int8')

assert X_train.columns.equals(X_test.columns)
assert X_train.notna().all().all() and X_test.notna().all().all()
print(f'Training: {len(X_train):,} shots across {train_shots.match_id.nunique():,} matches')
print(f'Test: {len(X_test):,} shots across {test_shots.match_id.nunique():,} matches')
print(f'First test date: {cutoff_date.date()}')
print(f'Predictors after encoding: {X_train.shape[1]}')


Training: 29,895 shots across 1,213 matches
Test: 7,548 shots across 304 matches
First test date: 2016-04-02
Predictors after encoding: 37


## Cross-validation for logistic regression

Use the earlier matches to try four values of `C`. For each of the five splits, train on earlier dates and validate on the next block of dates. Keep matches played on the same date together.

Learn the category groups, fill values, and scaling again from each split's training data. Pick the `C` with the lowest average log loss across the five validation blocks. Save the test matches for the final score.


In [5]:
shot_dates = pd.to_datetime(train_shots['match_date']).dt.normalize()
training_dates = pd.DatetimeIndex(shot_dates.unique()).sort_values()
date_splitter = TimeSeriesSplit(n_splits=5)
candidate_c_values = [0.01, 0.1, 1.0, 10.0]
fold_details = []
fold_results = []

for fold, (fit_indices, validation_indices) in enumerate(
    date_splitter.split(training_dates), start=1
):
    fit_dates = training_dates[fit_indices]
    validation_dates = training_dates[validation_indices]
    fold_train = train_shots.loc[shot_dates.isin(fit_dates)]
    fold_validation = train_shots.loc[shot_dates.isin(validation_dates)]

    assert fit_dates.max() < validation_dates.min()
    assert validation_dates.max() < cutoff_date
    assert set(fold_train['match_id']).isdisjoint(fold_validation['match_id'])
    fold_details.append({
        'fold': fold,
        'last_training_date': fit_dates.max().date(),
        'first_validation_date': validation_dates.min().date(),
        'last_validation_date': validation_dates.max().date(),
        'training_matches': fold_train['match_id'].nunique(),
        'validation_matches': fold_validation['match_id'].nunique(),
    })

    fitted = fit_shot_preprocessing(fold_train)
    X_fold_train = transform_shot_predictors(fold_train, fitted)
    X_fold_validation = transform_shot_predictors(fold_validation, fitted)
    y_fold_train = fold_train['is_goal'].astype('int8')
    y_fold_validation = fold_validation['is_goal'].astype('int8')

    for c_value in candidate_c_values:
        model = make_pipeline(
            StandardScaler(),
            LogisticRegression(C=c_value, max_iter=5000),
        )
        model.fit(X_fold_train, y_fold_train)
        probabilities = model.predict_proba(X_fold_validation)[:, 1]
        fold_results.append({
            'fold': fold,
            'C': c_value,
            'validation_shots': len(fold_validation),
            'log_loss': log_loss(y_fold_validation, probabilities, labels=[0, 1]),
        })

cv_folds = pd.DataFrame(fold_details)
cv_scores = pd.DataFrame(fold_results)
tuning_summary = (
    cv_scores.groupby('C', as_index=False)['log_loss'].mean()
    .rename(columns={'log_loss': 'mean_validation_log_loss'})
    .sort_values(['mean_validation_log_loss', 'C'], ignore_index=True)
)
best_c = float(tuning_summary.loc[0, 'C'])
display(cv_folds, tuning_summary)
print(f'Selected C: {best_c:g}')


,fold,last_training_date,first_validation_date,last_validation_date,training_matches,validation_matches
0,1,2015-09-23,2015-09-24,2015-11-01,226,210
1,2,2015-11-01,2015-11-02,2015-12-13,436,210
2,3,2015-12-13,2015-12-14,2016-01-17,646,179
3,4,2016-01-17,2016-01-18,2016-02-19,825,186
4,5,2016-02-19,2016-02-20,2016-04-01,1011,202


,C,mean_validation_log_loss
0,0.10,0.250701
1,1.00,0.250764
2,10.00,0.250787
3,0.01,0.251143


Selected C: 0.1


## Check the split

The test set should be similar to the training set, just with later matches. We want to make sure the goal rates and number of shots from each league aren't too different, since that could affect our model results.

In [6]:
# Compare the number of shots, matches and the goal rate in the train and test sets

print('Train shots:', len(train_shots), ' Matches:', train_shots['match_id'].nunique(), ' Goal rate:', round(train_shots['is_goal'].mean(), 3))
print('Test shots:', len(test_shots), ' Matches:', test_shots['match_id'].nunique(), ' Goal rate:', round(test_shots['is_goal'].mean(), 3))

Train shots: 29895  Matches: 1213  Goal rate: 0.093
Test shots: 7548  Matches: 304  Goal rate: 0.101


In [7]:
# Goal rate by league in the train set

print(train_shots.groupby('competition_name')['is_goal'].mean().round(3))

competition_name
La Liga           0.103
Ligue 1           0.096
Premier League    0.090
Serie A           0.086
Name: is_goal, dtype: float64


In [8]:
# Goal rate by league in the test set

print(test_shots.groupby('competition_name')['is_goal'].mean().round(3))

competition_name
La Liga           0.109
Ligue 1           0.105
Premier League    0.103
Serie A           0.091
Name: is_goal, dtype: float64


In [9]:
# Each league's share of the test shots

print(test_shots['competition_name'].value_counts(normalize=True).round(3))

competition_name
Serie A           0.280
Premier League    0.272
La Liga           0.242
Ligue 1           0.206
Name: proportion, dtype: float64


In [10]:
# Compare how close the shots were in the train and test sets

print('Train median distance:', round(train_shots['shot_distance'].median(), 1))
print('Test median distance:', round(test_shots['shot_distance'].median(), 1))

Train median distance: 19.3
Test median distance: 18.9


In [11]:
# Check that no match appears in both the train and test sets

overlap = set(train_shots['match_id']) & set(test_shots['match_id'])
print('Matches in both sets:', len(overlap))

Matches in both sets: 0


In [12]:
# Check that all training matches come before the test matches

print('Train dates:', train_shots['match_date'].min().date(), 'to', train_shots['match_date'].max().date())
print('Test dates:', test_shots['match_date'].min().date(), 'to', test_shots['match_date'].max().date())

Train dates: 2015-08-07 to 2016-04-01
Test dates: 2016-04-02 to 2016-05-17


In [13]:
# Share of shots with no linked receipt (the values we will fill in)

print('Train missing receipt:', round(train_shots['receipt_to_shot_seconds'].isna().mean(), 3))
print('Test missing receipt:', round(test_shots['receipt_to_shot_seconds'].isna().mean(), 3))

Train missing receipt: 0.28
Test missing receipt: 0.275


## Split check results

The training set has 29,895 shots from 1,213 matches, while the test set has 7,548 shots from 304 matches. There are no matches in both sets, and the split follows the season in order. Training covers August 2015 through April 1, 2016, and testing covers April 2 through May 17, 2016.

The two sets are fairly similar. The goal rate is slightly higher in the test set (10.1% versus 9.3%), and this increase appears in all four leagues. This suggests it may be a late-season pattern rather than something specific to one league. La Liga still has the highest goal rate and Serie A the lowest. Each league also makes up between 21% and 28% of the test set.

Test shots were slightly closer to goal, with a median distance of 18.9 coordinate units compared with 19.3 in training. This may explain some of the higher goal rate. Missing receipt values were also similar at 28.0% in training and 27.5% in testing.

Overall, the split looks reasonable. The higher goal rate in the test set could cause the model to underpredict goals, so we'll keep this in mind when checking calibration.

### Compare options for the missing receipt values

In [14]:
# Compare three ways of handling the missing receipt values
# 1. Fill with the training median and add a missing flag
# 2. Fill with zero and add a missing flag
# 3. Leave the two receipt features out

imputation_results = []

for fold, (fit_indices, validation_indices) in enumerate(date_splitter.split(training_dates), start=1):

    # Split the training matches into this fold's training and validation dates

    fold_train = train_shots[shot_dates.isin(training_dates[fit_indices])]
    fold_validation = train_shots[shot_dates.isin(training_dates[validation_indices])]

    # Learn category levels from training shots and set receipt fill values to zero.

    levels, receipt_zeros = fit_shot_preprocessing(fold_train)

    # Medians from this fold's training shots only, for the median option

    receipt_medians = fold_train[receipt_features].median()

    options = {
        'median + flag': (levels, receipt_medians),
        'zero + flag': (levels, receipt_zeros),
        'drop receipt features': (levels, receipt_zeros),
    }

    for option, fitted in options.items():
        X_fit = transform_shot_predictors(fold_train, fitted)
        X_val = transform_shot_predictors(fold_validation, fitted)

        # For the drop option, remove the two receipt features and the missing flag

        if option == 'drop receipt features':
            drop_columns = receipt_features + ['receipt_measurements_missing']
            X_fit = X_fit.drop(columns=drop_columns)
            X_val = X_val.drop(columns=drop_columns)

        model = make_pipeline(StandardScaler(), LogisticRegression(C=best_c, max_iter=5000))
        model.fit(X_fit, fold_train['is_goal'].astype(int))
        probabilities = model.predict_proba(X_val)[:, 1]

        imputation_results.append({
            'fold': fold,
            'option': option,
            'log_loss': log_loss(fold_validation['is_goal'].astype(int), probabilities),
            'brier': brier_score_loss(fold_validation['is_goal'].astype(int), probabilities),
        })

imputation_df = pd.DataFrame(imputation_results)
print(imputation_df.groupby('option')[['log_loss', 'brier']].mean().round(4))

                       log_loss   brier
option                                 
drop receipt features    0.2506  0.0712
median + flag            0.2507  0.0712
zero + flag              0.2507  0.0712


### Use zero and a flag for missing receipt values

Fill missing receipt time and displacement with zero. Set receipt_measurements_missing to 1 when either original value was missing, and 0 when both were recorded. A measured zero still has a flag of 0.

All three options performed the same in cross-validation, so we use zero plus the flag. It needs nothing learned from the training data, and the flag keeps shots without a receipt clearly separate.


In [15]:
# Prepare the model inputs with zero filling and the missing flag.
levels, _ = fit_shot_preprocessing(train_shots)
receipt_fill_values = pd.Series(0.0, index=receipt_features)
shot_preprocessing = (levels, receipt_fill_values)

X_train = transform_shot_predictors(train_shots, shot_preprocessing)
X_test = transform_shot_predictors(test_shots, shot_preprocessing)

# Check the flag against the original missing values.
for source, predictors in [(train_shots, X_train), (test_shots, X_test)]:
    missing = source[receipt_features].isna()
    expected_flag = missing.any(axis=1).astype('int8')
    assert predictors['receipt_measurements_missing'].equals(expected_flag)
    assert predictors[receipt_features].where(missing, 0).eq(0).all().all()
    for column in receipt_features:
        recorded = source[column].notna()
        assert predictors.loc[recorded, column].equals(source.loc[recorded, column])

assert X_train.columns.equals(X_test.columns)
assert X_train.notna().all().all() and X_test.notna().all().all()
print('Training shots flagged:', int(X_train['receipt_measurements_missing'].sum()))
print('Test shots flagged:', int(X_test['receipt_measurements_missing'].sum()))


Training shots flagged: 8377
Test shots flagged: 2073


## Leave-one-league-out split

Separate from the chronological holdout, this tests whether the model transfers to a league it was not trained on. The splits use the training matches only, so the final test matches stay untouched.

In [16]:
# Leave-one-league-out splits, built from the training matches only
# The final test matches (from April 2) stay untouched

leagues = ['La Liga', 'Premier League', 'Serie A', 'Ligue 1']

league_split_rows = []

for held_out_league in leagues:

    # Train on three leagues, test on the fourth

    league_train_shots = train_shots[train_shots['competition_name'] != held_out_league]
    league_test_shots = train_shots[train_shots['competition_name'] == held_out_league]

    # Save the size and goal rate of each side

    league_split_rows.append({'held_out_league': held_out_league,
                              'train_shots': len(league_train_shots),
                              'test_shots': len(league_test_shots),
                              'train_goal_rate': round(league_train_shots['is_goal'].mean(), 3),
                              'test_goal_rate': round(league_test_shots['is_goal'].mean(), 3)})

league_split_df = pd.DataFrame(league_split_rows)

league_split_df

,held_out_league,train_shots,test_shots,train_goal_rate,test_goal_rate
0,La Liga,22666,7229,0.090,0.103
1,Premier League,22148,7747,0.094,0.090
2,Serie A,22136,7759,0.096,0.086
3,Ligue 1,22735,7160,0.092,0.096


## Leave-one-league-out check

Each check uses the 29,895 training shots. One league (24% to 26% of the shots) is held out, and the model will be trained on the other three.

The biggest differences in goal rates are in La Liga (10.3% compared with 9.0% in the other leagues) and Serie A (8.6% compared with 9.6%). Because of this, the model might underpredict goals in La Liga and overpredict them in Serie A.

These two leagues will be especially useful for seeing how well the model works on data from a league it wasn't trained on. We'll report these results separately from the chronological test set.